# Predicting `arrears_next_month`
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, choose by mean CV ROC AUC, score holdout once).

**Feature decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. it is not known at prediction time. It is a leak (arrears rate 87% vs 1.4%) and is excluded, along with `meter_ref` and `billing_month` (per protocol).

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
TARGET='arrears_next_month'
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']
CAT=['tariff']; NUM=[f for f in FEATURES if f not in CAT]
train=df.iloc[train_idx]; hold=df.iloc[hold_idx]
print(len(train),len(hold),train.billing_month.min(),train.billing_month.max(),hold.billing_month.min())

## Candidates (named before any scoring)

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
candidates={
 'baseline_prior':Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'hgb_depth3':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'hgb_depth6':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=0))]),
}
list(candidates)

## Walk-forward CV by billing month (expanding window, 5 splits, training rows only)

In [ ]:
months=np.array(sorted(train.billing_month.unique()))
blocks=np.array_split(months[len(months)-20:],5)   # 5 consecutive test blocks; train = all earlier months
cv_folds=[]
for b in blocks:
    te=np.where(train.billing_month.isin(b))[0]
    tr=np.where(train.billing_month<b[0])[0]
    cv_folds.append((tr,te))
for tr,te in cv_folds: print(len(tr),len(te))
Xtr=train[FEATURES]; ytr=train[TARGET].to_numpy()
cv_scores={}
for name,pipe in candidates.items():
    s=[]
    for tr,te in cv_folds:
        m=clone(pipe).fit(Xtr.iloc[tr],ytr[tr])   # preprocessing fitted inside the fold
        s.append(float(roc_auc_score(ytr[te],m.predict_proba(Xtr.iloc[te])[:,1])))
    cv_scores[name]=s
cv_table=pd.DataFrame(cv_scores).T; cv_table['mean']=cv_table.mean(axis=1); cv_table

## Choose by mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
final=clone(candidates[chosen]).fit(Xtr,ytr)
holdout_auc=float(roc_auc_score(hold[TARGET],final.predict_proba(hold[FEATURES])[:,1]))
print(chosen,'CV mean',np.mean(cv_scores[chosen]),'HOLDOUT ROC AUC',holdout_auc)

In [ ]:
evidence={'features':FEATURES,'candidates':list(candidates),
 'cv_folds':[[[int(train_idx[i]) for i in tr],[int(train_idx[i]) for i in te]] for tr,te in cv_folds],
 'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':[int(i) for i in hold_idx],
 'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))